# Backend-7 (단발 실행) — Total 조립 불량 손실 시간 (prod_day=20260130) [FIXED]

이 노트북은 **스키마/테이블 대소문자 문제(따옴표 미사용 시 소문자 변환)** 로 인해
`relation does not exist`가 발생하는 케이스를 방지하기 위해,

- `information_schema.tables`에서 **실제 존재하는 스키마/테이블 이름을 자동 탐지**
- SQL에는 항상 `"Schema"."Table"` 형태로 **quoted FQN**을 사용

하도록 수정된 최종본입니다.

## 사양 반영(사용자 확정)
- 단발 실행, `prod_day="20260130"` 고정(텍스트 유지)
- 시간 정규화: `.5`는 **half-up** 반올림
- `wasted_time`은 **DB 값을 신뢰**, 분할은 정규화 시간으로 계산
- 분할 시간 계산: (from, to]의 **초 tick**을 카운트(예: 08:28:59~08:30:00 → night 60초, day 1초)
- 제품개수(행): 분할된 초가 **더 많은 shift에 1개 배정**, 동률이면 **day 우선**
- 경계 넘어가도 **분할 로직은 초만큼 해당 shift로 넘기면 충분**(일반화된 교차 계산으로 처리)
- `final_ct`: 이전달(month) + station='whole' + remark='PD' **1개만 존재 가정**
- 출력: `"h시간 m분 s초"` 형식(0단위 제외), `updated_at`은 KST timestamptz


In [1]:
# (필요 시) 설치
# !pip -q install sqlalchemy psycopg2-binary pandas python-dateutil

from __future__ import annotations

import os
import re
from datetime import datetime, date, timedelta
from decimal import Decimal, ROUND_HALF_UP
from zoneinfo import ZoneInfo

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

KST = ZoneInfo("Asia/Seoul")

# ===== 고정 입력 =====
PROD_DAY = "20260130"   # YYYYMMDD (text 유지)

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

# ===== 논리명(선호) =====
AFA_SCHEMA = "d1_machine_log"
AFA_TABLE  = "afa_fail_wasted_time"

CT_SCHEMA  = "e1_FCT_ct"     # 실제 DB에선 대소문자 섞여 있을 수 있음
CT_TABLE   = "fct_whole_op_ct"


In [2]:
def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )


# ---------- identifier quoting / table resolving (case-safe) ----------
_IDENT_RE = re.compile(r"^[A-Za-z0-9_]+$")

def quote_ident(name: str) -> str:
    if not _IDENT_RE.match(name):
        raise ValueError(f"Unsafe identifier: {name!r}")
    return f'"{name}"'


def resolve_table_fqn(engine: Engine, preferred_schema: str, preferred_table: str) -> str:
    """Find actual (schema, table) by case-insensitive match and return quoted FQN."""
    q = text("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE lower(table_name) = lower(:table_name)
          AND lower(table_schema) = lower(:schema_name)
        LIMIT 1
    """)
    with engine.begin() as conn:
        row = conn.execute(q, {"schema_name": preferred_schema, "table_name": preferred_table}).fetchone()

        if row is None:
            # schema가 다를 수 있으니 table_name만으로 fallback 탐색
            q2 = text("""
                SELECT table_schema, table_name
                FROM information_schema.tables
                WHERE lower(table_name) = lower(:table_name)
                ORDER BY table_schema, table_name
                LIMIT 1
            """)
            row = conn.execute(q2, {"table_name": preferred_table}).fetchone()

    if row is None:
        raise RuntimeError(
            f"Table not found (case-insensitive). preferred={preferred_schema}.{preferred_table}\n"
            f"→ DB에 {preferred_table} 테이블 존재 여부/권한 확인 필요"
        )

    schema_actual, table_actual = row[0], row[1]
    return f"{quote_ident(schema_actual)}.{quote_ident(table_actual)}"


In [3]:
def parse_yyyymmdd(s: str) -> date:
    return date(int(s[0:4]), int(s[4:6]), int(s[6:8]))


def shift_windows(prod_day: str) -> dict[str, tuple[datetime, datetime]]:
    d = parse_yyyymmdd(prod_day)
    day_start   = datetime(d.year, d.month, d.day, 8, 30, 0, tzinfo=KST)
    day_end     = datetime(d.year, d.month, d.day, 20, 29, 59, tzinfo=KST)
    night_start = datetime(d.year, d.month, d.day, 20, 30, 0, tzinfo=KST)
    d1 = d + timedelta(days=1)
    night_end   = datetime(d1.year, d1.month, d1.day, 8, 29, 59, tzinfo=KST)
    return {"day": (day_start, day_end), "night": (night_start, night_end)}


def prev_month_by_window_start(prod_day: str, shift: str) -> str:
    """[window 기준] 이전달: window_start가 속한 달 기준으로 -1 month."""
    from dateutil.relativedelta import relativedelta
    ws, _ = shift_windows(prod_day)[shift]
    prev = (ws.date().replace(day=1) - relativedelta(months=1))
    return f"{prev.year:04d}{prev.month:02d}"


def _time_to_decimal_seconds(t: str) -> Decimal:
    """HH:MM:SS or HH:MM:SS.xx -> total seconds as Decimal."""
    hh, mm, ss = t.split(":")
    return Decimal(hh) * 3600 + Decimal(mm) * 60 + Decimal(ss)


def normalize_time_to_dt(end_day: str, t: str) -> datetime:
    """.5 half-up rounding to seconds. Carry over day if rounded to >= 24:00:00."""
    base = parse_yyyymmdd(end_day)
    sec = _time_to_decimal_seconds(t)
    sec_i = int(sec.quantize(Decimal("1"), rounding=ROUND_HALF_UP))

    day_add = 0
    if sec_i >= 86400:
        day_add = sec_i // 86400
        sec_i = sec_i % 86400
    if sec_i < 0:
        # 방어 (거의 없음)
        day_add = -((-sec_i + 86399) // 86400)
        sec_i = sec_i % 86400

    hh = sec_i // 3600
    mm = (sec_i % 3600) // 60
    ss = sec_i % 60
    dt = datetime(base.year, base.month, base.day, hh, mm, ss, tzinfo=KST) + timedelta(days=day_add)
    return dt


def overlap_seconds_tick(from_dt: datetime, to_dt: datetime, ws: datetime, we: datetime) -> int:
    """Count seconds in intersection of (from_dt, to_dt] and [ws, we] using 1-second ticks."""
    if to_dt <= from_dt:
        return 0
    start_tick = from_dt + timedelta(seconds=1)  # open at from
    end_tick = to_dt  # closed at to
    if start_tick > end_tick:
        return 0

    lo = max(start_tick, ws)
    hi = min(end_tick, we)
    if lo > hi:
        return 0
    return int((hi - lo).total_seconds()) + 1


def format_kor_hms(total_seconds: int) -> str:
    total_seconds = max(0, int(total_seconds))
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    parts = []
    if h:
        parts.append(f"{h}시간")
    if m:
        parts.append(f"{m}분")
    if s or (not parts):
        parts.append(f"{s}초")
    return " ".join(parts)


In [4]:
# 0) DB 연결 + 실제 테이블 FQN 확정(대소문자/따옴표 이슈 방지)
engine = make_engine()

AFA_FQN = resolve_table_fqn(engine, AFA_SCHEMA, AFA_TABLE)
CT_FQN  = resolve_table_fqn(engine, CT_SCHEMA, CT_TABLE)

print("AFA_FQN:", AFA_FQN)
print("CT_FQN :", CT_FQN)

# window 설정
win = shift_windows(PROD_DAY)
(day_ws, day_we) = win["day"]
(nig_ws, nig_we) = win["night"]

d = parse_yyyymmdd(PROD_DAY)
d1 = d + timedelta(days=1)
END_DAYS = [d.strftime("%Y%m%d"), d1.strftime("%Y%m%d")]

print("DAY  window:", day_ws, "~", day_we)
print("NIGHT window:", nig_ws, "~", nig_we)
print("QUERY end_day IN:", END_DAYS)


AFA_FQN: "d1_machine_log"."afa_fail_wasted_time"
CT_FQN : "e1_FCT_ct"."fct_whole_op_ct"
DAY  window: 2026-01-30 08:30:00+09:00 ~ 2026-01-30 20:29:59+09:00
NIGHT window: 2026-01-30 20:30:00+09:00 ~ 2026-01-31 08:29:59+09:00
QUERY end_day IN: ['20260130', '20260131']


In [5]:
# 1) afa_fail_wasted_time 로드 (end_day는 from/to가 속한 날짜이며, 자정 넘김은 end_day가 2개로 표현됨)
sql_afa = text(f"""
SELECT end_day, from_time, to_time, wasted_time
FROM {AFA_FQN}
WHERE end_day = ANY(:end_days)
""")

with engine.begin() as conn:
    afa = pd.read_sql(sql_afa, conn, params={"end_days": END_DAYS})

print("rows:", len(afa))
afa.head()


rows: 2


,end_day,from_time,to_time,wasted_time
0,20260130,09:13:31.00,09:14:03.00,32.0
1,20260130,15:29:04.00,15:29:25.00,21.0


In [6]:
# 2) row별 day/night 손실시간(초) 배분 + 제품개수(행) 배정
day_loss = 0
night_loss = 0
day_cnt = 0
night_cnt = 0

afa["wasted_time"] = pd.to_numeric(afa["wasted_time"], errors="coerce").fillna(0).astype(int)

for r in afa.itertuples(index=False):
    end_day = str(r.end_day)
    from_t = str(r.from_time)
    to_t = str(r.to_time)
    wasted = int(r.wasted_time)

    # 시간 정규화(.5 half-up)
    fdt = normalize_time_to_dt(end_day, from_t)
    tdt = normalize_time_to_dt(end_day, to_t)

    # to < from 이면 자정 넘김으로 보정(방어)
    if tdt < fdt:
        tdt = tdt + timedelta(days=1)

    # (from,to] tick 기준으로 shift별 겹치는 '초' 계산
    sec_day = overlap_seconds_tick(fdt, tdt, day_ws, day_we)
    sec_nig = overlap_seconds_tick(fdt, tdt, nig_ws, nig_we)
    sec_total = sec_day + sec_nig
    if sec_total <= 0:
        continue

    # wasted_time은 DB 값을 신뢰하되, 분할 기준은 overlap 비율 사용
    if wasted <= 0:
        alloc_day, alloc_nig = 0, 0
    elif sec_total == wasted:
        alloc_day, alloc_nig = sec_day, sec_nig
    else:
        # 비율 배분 (half-up) + 합은 wasted로 보존
        alloc_day = int((Decimal(wasted) * Decimal(sec_day) / Decimal(sec_total)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))
        alloc_day = max(0, min(wasted, alloc_day))
        alloc_nig = wasted - alloc_day

    day_loss += alloc_day
    night_loss += alloc_nig

    # 제품개수(행) 배정: 더 많은 '겹친 초' 쪽 (동률 day 우선)
    if sec_day >= sec_nig:
        day_cnt += 1
    else:
        night_cnt += 1

print("day_loss_sec:", day_loss, "day_cnt:", day_cnt)
print("night_loss_sec:", night_loss, "night_cnt:", night_cnt)


day_loss_sec: 53 day_cnt: 2
night_loss_sec: 0 night_cnt: 0


In [7]:
# 3) final_ct 로드 (이전달, station='whole', remark='PD') — 1개만 존재 가정
prev_month = prev_month_by_window_start(PROD_DAY, "day")  # day/night 모두 window_start는 PROD_DAY이므로 동일

sql_ct = text(f"""
SELECT final_ct
FROM {CT_FQN}
WHERE month = :month
  AND station = 'whole'
  AND remark = 'PD'
LIMIT 1
""")

with engine.begin() as conn:
    row = conn.execute(sql_ct, {"month": prev_month}).fetchone()

if row is None:
    raise RuntimeError(f"final_ct not found for month={prev_month}, station='whole', remark='PD'")

final_ct = float(row[0])  # double precision 가정
print("prev_month:", prev_month, "final_ct:", final_ct)


prev_month: 202512 final_ct: 9.82


In [8]:
# 4) 재작업 시간(초) + Total 조립 불량 손실 시간(초) 계산
# (사용자 확정) 재작업 시간 = final_ct × 제품개수(행)
# ※ 재작업 시간도 '초'로 반올림(half-up)하여 정수화 (예: 9.82*2=19.64 -> 20초)
day_rework = int((Decimal(str(final_ct)) * Decimal(day_cnt)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))
night_rework = int((Decimal(str(final_ct)) * Decimal(night_cnt)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))

day_total_sec = int(day_loss + day_rework)
night_total_sec = int(night_loss + night_rework)

print("day_rework_sec:", day_rework, "day_total_sec:", day_total_sec)
print("night_rework_sec:", night_rework, "night_total_sec:", night_total_sec)


day_rework_sec: 20 day_total_sec: 73
night_rework_sec: 0 night_total_sec: 0


In [10]:
# 5) 출력 DF 생성 (day/night 분리 출력)
updated_at = datetime.now(tz=KST)

day_df = pd.DataFrame(
    [{
        "prod_day": PROD_DAY,
        "shift_type": "day",
        "Total 조립 불량 손실 시간": format_kor_hms(day_total_sec),
        "updated_at": updated_at,
    }]
)

night_df = pd.DataFrame(
    [{
        "prod_day": PROD_DAY,
        "shift_type": "night",
        "Total 조립 불량 손실 시간": format_kor_hms(night_total_sec),
        "updated_at": updated_at,
    }]
)

print("=== DAY ===")
display(day_df)

print("=== NIGHT ===")
display(night_df)

=== DAY ===


,prod_day,shift_type,Total 조립 불량 손실 시간,updated_at
0,20260130,day,1분 13초,2026-02-01 10:03:15.851427+09:00


=== NIGHT ===


,prod_day,shift_type,Total 조립 불량 손실 시간,updated_at
0,20260130,night,0초,2026-02-01 10:03:15.851427+09:00


In [11]:
# 6) 결과 DF 저장 (i_daily_report.g_afa_wasted_time_day_daily / night_daily)

from sqlalchemy import text

SAVE_SCHEMA = "i_daily_report"
T_DAY   = "g_afa_wasted_time_day_daily"
T_NIGHT = "g_afa_wasted_time_night_daily"

def ensure_schema(engine: Engine, schema: str) -> None:
    with engine.begin() as conn:
        conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{schema}";'))

def ensure_table_for_df(engine: Engine, schema: str, table: str, df: pd.DataFrame) -> None:
    """
    - df 컬럼 기준으로 테이블 생성(없으면)
    - updated_at: timestamptz
    - 그 외: text
    """
    cols_ddl = []
    for c in df.columns:
        if c == "updated_at":
            cols_ddl.append(f'"{c}" timestamptz')
        else:
            cols_ddl.append(f'"{c}" text')
    ddl = ",\n  ".join(cols_ddl)

    with engine.begin() as conn:
        conn.execute(text(f'''
            CREATE TABLE IF NOT EXISTS "{schema}"."{table}" (
              {ddl}
            );
        '''))

def replace_table(engine: Engine, schema: str, table: str, df: pd.DataFrame) -> None:
    """
    단발 리포트 기준:
    - 해당 테이블을 TRUNCATE 후
    - df 전체 재적재
    """
    with engine.begin() as conn:
        conn.execute(text(f'TRUNCATE TABLE "{schema}"."{table}";'))

    # pandas to_sql 은 schema/table에 자동 quoting이 애매할 수 있어,
    # 스키마/테이블 이름은 소문자 안전 케이스로만 사용. (현재는 소문자+언더스코어라 OK)
    df.to_sql(table, engine, schema=schema, if_exists="append", index=False, method="multi")

# ---- 실행 ----
ensure_schema(engine, SAVE_SCHEMA)

# day 저장
ensure_table_for_df(engine, SAVE_SCHEMA, T_DAY, day_df)
replace_table(engine, SAVE_SCHEMA, T_DAY, day_df)
print(f"[SAVED] day_df -> {SAVE_SCHEMA}.{T_DAY}")

# night 저장
ensure_table_for_df(engine, SAVE_SCHEMA, T_NIGHT, night_df)
replace_table(engine, SAVE_SCHEMA, T_NIGHT, night_df)
print(f"[SAVED] night_df -> {SAVE_SCHEMA}.{T_NIGHT}")

[SAVED] day_df -> i_daily_report.g_afa_wasted_time_day_daily
[SAVED] night_df -> i_daily_report.g_afa_wasted_time_night_daily
